## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [1]:
# -*- coding: utf8 -*-
import codecs,glob
import features
import re,os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import pickle
#%pylab inline
#pd.options.display.mpl_style = 'default'
debug=False
from __future__ import print_function
from IPython.display import display
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [2]:
pd.__version__
debug=False
debug1=False


In [3]:
phonologicalMap="-X"

In [4]:
rep="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"

In [5]:
import math
def rAn(r,n):
    f = math.factorial
    return f(n) / f(n-r)
def rCn(r,n):
    f = math.factorial
    return f(n) / f(r) / f(n-r)

### Préparation des matrices de traits

In [6]:
features.add_config('/Users/gilles/Github/SWIM/ParadigmGeneration/Vlexique2/bdlexique.ini')
fs=features.FeatureSystem('phonemes')

In [7]:
validPhonemes=list(fs.supremum.concept.extent)
for phoneme in validPhonemes:
    print (phoneme, [phoneme], ";", end=" ")

p ['p'] ; t ['t'] ; k ['k'] ; b ['b'] ; d ['d'] ; g ['g'] ; f ['f'] ; s ['s'] ; S ['S'] ; v ['v'] ; z ['z'] ; Z ['Z'] ; m ['m'] ; n ['n'] ; J ['J'] ; N ['N'] ; j ['j'] ; l ['l'] ; r ['r'] ; w ['w'] ; H ['H'] ; i ['i'] ; y ['y'] ; E ['E'] ; e ['e'] ; 9 ['9'] ; 2 ['2'] ; 6 ['6'] ; a ['a'] ; u ['u'] ; O ['O'] ; o ['o'] ; ê ['ê'] ; û ['û'] ; â ['â'] ; ô ['ô'] ; 

In [8]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

neutralisationsTotales=(u"e2o6û",u"E9O9ê")
totalNeutreIn=u"èò"+neutralisationsTotales[0]
totalNeutreNum=[ord(char) for char in totalNeutreIn]
totalNeutreOut=u"EO"+neutralisationsTotales[1]
totalNeutralise = dict(zip(totalNeutreNum, totalNeutreOut))

In [9]:
def recoder(chaine,table=totalNeutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [10]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}

In [11]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        m=re.match(r"^.*([^ieèEaOouy926êôâ])[jwH]$",result)
        if m:
            print ("pb avec un glide final", [prononciation])
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
    else:
        result=prononciation
    return result

In [12]:
checkFrench("ye")

'yHE'

# Préparation du calcul des analogies

### Calcul de la différence entre deux formes

In [13]:
def diff(mot1,mot2):
    result=[]
    diff1=""
    diff2=""
    same=""
    vide="."
    lmax=max(len(mot1),len(mot2))
    lmin=min(len(mot1),len(mot2))
    for index in range(lmax):
        if index < lmin:
            if mot1[index]!=mot2[index]:
                diff1+=mot1[index]
                diff2+=mot2[index]
                same+=vide
            else:
                same+=mot1[index]
                diff1+=vide
                diff2+=vide
        elif index < len(mot1):
            diff1+=mot1[index]
        elif index < len(mot2):
            diff2+=mot2[index]
    diff1=diff1.lstrip(".")
    diff2=diff2.lstrip(".")
#    return (same,diff1,diff2,diff1+"_"+diff2)
    return (diff1+"-"+diff2)

### Accumulation des paires appartenant à un patron

In [14]:
def rowDiff(row, patrons):
    result=diff(row[0],row[1])
    if not result in patrons:
        patrons[result]=(formesPatron(),formesPatron())
    patrons[result][0].ajouterFormes(row[0])
    patrons[result][1].ajouterFormes(row[1])
    return (result[0],result[1])

### Transformation d'un patron en RegExp

In [15]:
def patron2regexp(morceaux):
    result="^"
    for morceau in morceaux:
        if morceau=="*":
            result+="(.*)"
        elif len(morceau)>1:
            result+="(["+morceau+"])"
        else:
            result+=morceau
    result+="$"
    result=result.replace(")(","")
    return result

### Substitution de sortie 
???

In [16]:
def remplacementSortie(sortie):
    n=1
    nsortie=""
    for lettre in sortie:
        if lettre==".":
            nsortie+="\g<%d>"%n
            n+=1
        else:
            nsortie+=lettre
    return nsortie

In [17]:
class formesPatron:
    '''
    Accumulateur de formes correspondant à un patron pour calcul de la Généralisation Minimale (cf. MGL)
    '''
    def __init__(self):
        self.formes=[]

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterForme(self,forme):
        self.formes.append(forme)
        
    def calculerGM(self):
        minLongueur=len(min(self.formes, key=len))
        maxLongueur=len(max(self.formes, key=len))
        if debug: 
            print (minLongueur, maxLongueur)
        positions=[]
        if maxLongueur>minLongueur:
            positions.append("*")
        for i in xrange(minLongueur, 0, -1):
            phonemes=set([x[-i] for x in self.formes])
            if debug: 
                print (phonemes)
            if "." in phonemes:
                positions.append(".")
            else:
                positions.append("".join(fs.lattice[phonemes].extent))
        return patron2regexp(positions)

class pairePatrons:
    '''
    Accumulateur de triplets (f1,f2,patron) correspondant à une paire pour calcul des Généralisations Minimales (cf. MGL)
    '''
    def __init__(self,case1,case2):
        self.patrons1={}
        self.patrons2={}
        self.case1=case1
        self.case2=case2

#    def __repr__(self):
#        return ','.join(self.calculerGM())
        
    def ajouterFormes(self,forme1,forme2,patron):
#        print (forme1,forme2,patron, file=logfile)
        patron12=patron
        (pat1,pat2)=patron.split("-")
        patron21=pat2+"-"+pat1
#        print (patron12,patron21, file=logfile)
        if not patron12 in self.patrons1:
            self.patrons1[patron12]=formesPatron()
        self.patrons1[patron12].ajouterForme(forme1)
        if not patron21 in self.patrons2:
            self.patrons2[patron21]=formesPatron()
        self.patrons2[patron21].ajouterForme(forme2)
        
        
    def calculerGM(self):
        resultat1={}
        for patron in self.patrons1:
            if debug: 
                print ("patron1", patron, file=logfile)
                print ("patron1", patron)
            resultat1[patron]=self.patrons1[patron].calculerGM()
        resultat2={}
        for patron in self.patrons2:
            if debug: 
                print ("patron2", patron, file=logfile)
                print ("patron2", patron)
            resultat2[patron]=self.patrons2[patron].calculerGM()
        return (resultat1,resultat2) 

# Classe pour la gestion des patrons, des classes et des transformations

In [18]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n, file=logfile)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

    
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=False):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug: 
                    print (forme, file=logfile)
                    print ("pas de classe",idClasseForme, file=logfile)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)), file=logfile)
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme, file=logfile) 
                print ("pas de patron", file=logfile)
        return sortieForme
        

## Appliquer la formule de calcul des différences entre chaines à chaque ligne

>si il y a au moins une ligne

>>on applique la différence à la ligne

>>on calcule les deux patrons par suppression des points initiaux

>>on renvoie le groupement par patrons (1&2)

>sinon

>>on renvoie le paradigme vide d'origine

In [19]:
def rapports(paradigme):
    (lexeme,case1,case2)= paradigme.columns.values.tolist()
    patrons=pairePatrons(case1,case2)
    classes=paireClasses(case1,case2)
    # if case1 in [u'ai3S', u'inf'] and case2 in [u'ai3S', u'inf']:
    #     print (patrons.patrons1)
    #     print (patrons.patrons2)
        # print (classes)
    if len(paradigme)>0:
        paradigme.apply(lambda x: patrons.ajouterFormes(x.iloc[1],x.iloc[2],diff(x.iloc[1],x.iloc[2])), axis=1)
        (regles1,regles2)=patrons.calculerGM()
        for regle in regles1:
            classes.ajouterPatron(1,regle,regles1[regle])
        for regle in regles2:
            classes.ajouterPatron(2,regle,regles2[regle])
        paradigme.apply(lambda x: classes.ajouterPaire(x.iloc[1],x.iloc[2]), axis=1)
    (classes1,classes2)=classes.calculerClasses()
    return (classes1,classes2)

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

In [20]:
def splitCellMates(df,colonne):
    '''
    Calcul d'une dataframe sans surabondance par dédoublement des valeurs
    '''
    dfPaire=df.copy()
    indexCols=dfPaire.columns.tolist()
    indexCols.remove(colonne)
    # print(indexCols)
    dfPaire=dfPaire.set_index(indexCols).apply(lambda x: x.str.split(',').explode()).reset_index()
    return dfPaire[indexCols+[colonne]]


## Calculer les rapports entre formes pour chaque paire

>on fait la liste des cases de *paradigmes*

>pour chaque paire du tableau principal

>>si la paire fait partie des cases de *paradigmes*

>>>on calcule le rapport

>>sinon

>>>on signale que qu'une des cases n'est pas représentée

In [21]:
def evaluerEchantillon(paradigmes):
    result={}
    colonnes=paradigmes.columns.values.tolist()
    for n,paire in enumerate(it.combinations_with_replacement(sampleCases,2)):
        if debug: print (paire)
        if debug: print ("-".join(paire),end=", ")
        paireListe=list(paire)
        paireListe.insert(0,"lexeme")
        if paire[0] in colonnes and paire[1] in colonnes:
            paradigmePaire=paradigmes[paireListe].dropna(thresh=3, axis=0).reindex()
            if paire[0]==paire[1]:
                paradigmePaire=paradigmes[paireListe].dropna(thresh=2, axis=0).reindex()
                paradigmePaire.columns=["lexeme", paire[0],paire[0]+"-bis"]
                paradigmePaire=splitCellMates(splitCellMates(paradigmePaire,paire[0]),paire[0]+"-bis")
                paradigmePaire.columns=["lexeme", paire[0],paire[0]]
                # if paire[0]=="inf": print(paradigmePaire.to_string())
            else:
                paradigmePaire=splitCellMates(splitCellMates(paradigmePaire,paire[0]),paire[1])
            # display(paradigmePaire)
            result[paire]=rapports(paradigmePaire)
            # if paire[0]=="inf" and paire[1]=="inf":
                # (a,b)=result[paire]
                # print(a.patrons)
                # print(b.patrons)
        else:
            result[paire]=("missing pair", paire)
    return result

### Boucle de calcul des analogies pour l'échantillon

In [22]:
inputFiles=["vlexique2-S%d-omp.csv"%n for n in range(9)]
inputFiles+=["vlexique2-CV%d-%s%d-omp.csv"%(n,t,i) for n in [2,3,5,10] for t in ["Train","Test"] for i in range(n)]
inputFiles


for fTirages in inputFiles[:]:

    nomLexique=rep+fTirages
    paradigmes=pd.read_csv(nomLexique,sep=";")
    print(fTirages)
    display(paradigmes.head())

    casesPrincipales= [
            'inf', 'pi1S', 'pi2S', 'pi3S', 'pi1P', 'pi2P', 'pi3P', 'ii1S',
            'ii2S', 'ii3S', 'ii1P', 'ii2P', 'ii3P', 
            'fi1S', 'fi2S', 'fi3S', 'fi1P', 'fi2P',
            'fi3P', 'pI2S', 'pI1P', 'pI2P', 'ps1S', 'ps2S', 'ps3S', 'ps1P',
            'ps2P', 'ps3P', 
            'pc1S', 'pc2S', 'pc3S', 'pc1P', 'pc2P', 'pc3P', 'pP',
            'ppMS', 'ppMP', 'ppFS', 'ppFP'
                ]
    casesSecondaires= [
           'ai1S', 'ai2S', 'ai3S', 'ai1P', 'ai2P', 'ai3P', 'is1S', 'is2S', 'is3S', 'is1P', 'is2P', 'is3P'
                ]
    casesTotales=casesPrincipales+casesSecondaires
    listeCases=casesTotales    

    if u"Unnamed: 0" in paradigmes:
        del paradigmes[u"Unnamed: 0"]
    if debug: print(paradigmes.columns)

    sampleCases=paradigmes.columns.tolist()
    sampleCases.remove(u"lexeme")
    if debug: print(",".join(sampleCases))
    if debug: print("nombre de cases du paradigme de l'échantillon :",len(sampleCases))

    #Adapt all the forms to French phonology
    for case in sampleCases:
        paradigmes[case]=paradigmes[case].apply(lambda x: checkFrench(x))
    
    # in python3 xrange a changé de nom pour range.
    xrange=range
    
    resultats=evaluerEchantillon(paradigmes)

    classesFinales={}
    for resultat in resultats:
        classesFinales[resultat]=resultats[resultat][0]
        classesFinales[(resultat[1],resultat[0])]=resultats[resultat][1]

    with open(rep+fTirages.replace(".csv","-Regles.pkl"), 'wb') as output:
       pickle.dump(classesFinales, output, pickle.HIGHEST_PROTOCOL)
    


vlexique2-S2-omp.csv


,lexeme,ai1P,ai1S,ai2P,ai3P,fi1P,fi1S,fi2P,fi3P,ii1P,...,ps2P,ps3S,ii3S,ai3S,ppFP,ppMS,is1S,pc3S,fi3S,pi3S
0,abaisser,NaN,NaN,NaN,NaN,abEs6rô,abEs6rE,NaN,abEs6rô,NaN,...,NaN,abEs,abEsE,abEsa,abEse,abEse,NaN,abEs6rE,abEs6ra,abEs
1,abandonner,NaN,abâdOnE,NaN,abâdOnEr,abâdOn6rô,abâdOn6rE,abâdOn6re,abâdOn6rô,abâdOnjô,...,abâdOnje,abâdOn,abâdOnE,abâdOna,abâdOne,abâdOne,NaN,abâdOn6rE,abâdOn6ra,abâdOn
2,abasourdir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,abazurdi,abazurdi,NaN,NaN,NaN,abazurdi
3,abattre,NaN,NaN,NaN,abatir,abatrô,abatrE,abatre,abatrô,NaN,...,abatje,abat,abatE,abati,abaty,abaty,NaN,abatrE,abatra,aba
4,abdiquer,NaN,NaN,NaN,NaN,NaN,abdik6rE,abdik6re,NaN,NaN,...,NaN,abdik,NaN,abdika,NaN,abdike,NaN,NaN,abdik6ra,abdik


vlexique2-S3-omp.csv


,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi2P,fi3P,...,pi3P,ps1P,ps2P,pc3P,is2S,ii3S,ppFP,ppMS,fi3S,pi3S
0,abaisser,NaN,NaN,NaN,NaN,NaN,abEsa,abEs6rô,NaN,abEs6rô,...,abEs,NaN,NaN,abEs6rE,abEs,abEsE,abEse,abEse,abEs6ra,abEs
1,abandonner,NaN,abâdOnE,NaN,NaN,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6re,abâdOn6rô,...,abâdOn,abâdOnjô,abâdOnje,abâdOn6rE,abâdOn,abâdOnE,abâdOne,abâdOne,abâdOn6ra,abâdOn
2,abasourdir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,abazurdi,abazurdi,NaN,NaN
3,abattre,NaN,NaN,NaN,NaN,abatir,abati,abatrô,abatre,abatrô,...,abat,NaN,abatje,abatrE,abat,abatE,abaty,abaty,abatra,aba
4,abdiquer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abdik6rE,abdik,NaN,NaN,abdike,abdik6ra,abdik


vlexique2-S4-omp.csv


...

,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1S,fi2P,ii1P,...,ps1P,pi3P,ii3S,pi3S,ppFP,fi3P,pi1P,pI2P,ps2P,fi3S
0,abaisser,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,abEsjô,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abandonner,NaN,NaN,abâdOnat,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,abâdOnE,abâdOn,NaN,NaN,abâdOnô,abâdOne,NaN,NaN
2,abasourdir,NaN,NaN,NaN,NaN,NaN,NaN,abazurdirE,NaN,NaN,...,NaN,NaN,NaN,NaN,abazurdi,NaN,NaN,NaN,NaN,NaN
3,abattre,NaN,NaN,NaN,abati,NaN,NaN,NaN,NaN,abatjô,...,NaN,abat,abatE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,abdiquer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abdik6re,NaN,...,NaN,NaN,abdikE,NaN,NaN,abdik6rô,NaN,NaN,NaN,NaN


vlexique2-CV10-Test8-omp.csv


,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,pi3S,ppMP,ps1P,ps3S,ii3S,pi2S,ppFP,pi2P,ii2P,pi1P
0,abaisser,NaN,abEsE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abEs,abEsE,NaN,NaN,NaN,NaN,NaN
1,abandonner,NaN,NaN,NaN,NaN,NaN,NaN,abâdOn6rô,NaN,NaN,...,NaN,abâdOne,NaN,NaN,NaN,NaN,NaN,NaN,abâdOnje,NaN
2,abattre,NaN,NaN,NaN,NaN,NaN,abati,NaN,abatrE,NaN,...,NaN,NaN,NaN,abat,NaN,NaN,NaN,NaN,NaN,NaN
3,abdiquer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,abhorrer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,abOr,NaN,NaN,NaN,NaN,NaN,abOre,NaN,NaN,NaN


vlexique2-CV10-Test9-omp.csv


,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1S,fi2P,ii1P,...,ps2P,ps2S,ps3P,ps3S,pI2S,ii3S,fi3P,pi2P,pi1P,fi3S
0,abaisser,NaN,NaN,NaN,NaN,NaN,abEsa,NaN,abEs6re,NaN,...,NaN,NaN,NaN,NaN,NaN,abEsE,abEs6rô,NaN,NaN,NaN
1,abandonner,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abâdOn6re,NaN,...,NaN,NaN,abâdOn,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,abasourdir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abazurdis,NaN,NaN,NaN,abazurdise,NaN,NaN
3,abattre,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abate,NaN,NaN
4,abdiquer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,abdik6rô,NaN,NaN,NaN


# Fin du traitement